# CIFAR-10 CLAQ Experiment

A notebook for the CIFAR-10 CLAQ workflow:

- load config, CLIP, concepts, and data
- load or train Concept-QA
- load or train baseline and CLAQ
- optionally compare `baseline` vs `lam=0.4` with fixed-history summary numbers
- probe the sensitive concept set before retraining
- check sample-level sensitive-label sanity
- replay tiny-start cases


In [1]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch

from claq.analysis import (
    evaluate_bundles_on_fixed_histories,
    plot_rollout_comparisons,
    sample_intuition_replays,
)
from claq.config import Cifar10ClaqConfig, default_paths
from claq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_concepts,
    load_run_bundle,
    make_sensitive_mask,
    save_bundle_checkpoint,
)
from claq.data import get_cifar10_datasets, get_cifar10_loaders, get_raw_cifar10_dataset
from claq.models import ConceptNet2
from claq.sensitive_labels import (
    build_cifar10_sensitive_match,
    build_sensitive_labels,
    load_sensitive_labels,
    save_sensitive_labels,
)
from claq.training import HistorySamplingConfig, build_claq_models, fit_concept_qa, fit_claq, seed_everything
from claq.training.concept_qa import load_gpt_answers

In [2]:
repo_root = Path.cwd().resolve()
if not (repo_root / "claq").exists() and (repo_root.parent / "claq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = Cifar10ClaqConfig()
device = config.device
seed_everything(config.random_seed)

# Change this to ".pdf" or ".png" when needed.
figure_ext = ".svg"


def figure_path(stem):
    return paths.figures_root / f"{stem}{figure_ext}"


print(f"repo_root: {repo_root}")
print(f"artifacts_root: {paths.artifacts_root}")
print(f"device: {device}")
print(f"figure_ext: {figure_ext}")

repo_root: /home/jupyter/claq
artifacts_root: /home/jupyter/claq/artifacts
device: cuda
figure_ext: .svg


In [3]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
concepts = load_concepts(paths.concept_file)
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sensitive_match = build_cifar10_sensitive_match(concepts)
sens_idx = sensitive_match.indices
sensitive_mask = make_sensitive_mask(config.max_queries, sens_idx, device)

train_ds, test_ds = get_cifar10_datasets(transform=preprocess, root=paths.data_root)
train_loader, test_loader = get_cifar10_loaders(
    transform=preprocess,
    root=paths.data_root,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
)
raw_test_ds = get_raw_cifar10_dataset(paths.data_root, train=False)

print(f"# concepts: {len(concepts)}")
print(f"# sensitive concepts matched: {len(sensitive_match.matched)}")
print(sensitive_match.matched)
if sensitive_match.missing:
    print(f"# sensitive concepts missing: {len(sensitive_match.missing)}")
    print(sensitive_match.missing)

100%|██████████| 170M/170M [1:04:14<00:00, 44.2kB/s] 


# concepts: 128
# sensitive concepts matched: 23
['a bridle', 'a cab for the driver', 'a captain', 'a collar', 'a copilot', 'a dashboard', 'a driver', 'a flight attendant', 'a gear shift', 'a halter', 'a hitch', 'a lead rope', 'a leash', 'a passenger', 'a pedal', 'a pilot', 'a reins', 'a rider', 'a rifle', 'a saddle', 'a seatbelt', 'a steering wheel', 'a trailer']


In [4]:
qa_checkpoint = paths.checkpoints_root / "concept_qa_cifar10.pt"
qa_source = qa_checkpoint
if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
elif paths.bootstrap_concept_qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(paths.bootstrap_concept_qa_checkpoint, device=device)
    qa_source = paths.bootstrap_concept_qa_checkpoint
elif paths.gpt_answers_file.exists():
    gpt_answers = load_gpt_answers(paths.gpt_answers_file)
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.SGD(qa_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(qa_optimizer, T_max=200)
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=train_loader,
        eval_loader=test_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=2,
        model_clip=model_clip,
        dictionary=dictionary,
        gpt_answers=gpt_answers,
        clip_device=device,
        train_device=device,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(paths.runs_root / "concept_qa_cifar10_history.json", "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()
else:
    raise FileNotFoundError("No local Concept-QA checkpoint, bootstrap checkpoint, or GPT answers file available.")

print(f"Concept-QA ready from: {qa_source}")

Concept-QA ready from: /home/jupyter/claq/artifacts/models/bootstrap/concept_qa_cifar10_reference.pth


In [5]:
sensitive_labels_dir = paths.sensitive_labels_root

label_files = [
    sensitive_labels_dir / "s_soft_train.npy",
    sensitive_labels_dir / "s_hard_train.npy",
    sensitive_labels_dir / "s_soft_test.npy",
    sensitive_labels_dir / "s_hard_test.npy",
]

if all(path.exists() for path in label_files):
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "cache"
else:
    s_soft_train, s_hard_train = build_sensitive_labels(
        loader=train_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        sens_idx=sens_idx,
        clip_device=device,
        tau=config.sensitive_tau,
        topk=config.sensitive_topk,
        desc="Building sensitive labels (train)",
    )
    s_soft_test, s_hard_test = build_sensitive_labels(
        loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        sens_idx=sens_idx,
        clip_device=device,
        tau=config.sensitive_tau,
        topk=config.sensitive_topk,
        desc="Building sensitive labels (test)",
    )
    save_sensitive_labels(
        sensitive_labels_dir,
        train_soft=s_soft_train,
        train_hard=s_hard_train,
        test_soft=s_soft_test,
        test_hard=s_hard_test,
    )
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "built_and_saved"

print(f"Sensitive labels ready from: {label_source} -> {sensitive_labels_dir}")
print(
    {
        "s_soft_train": sensitive_label_cache["s_soft_train"].shape,
        "s_hard_train": sensitive_label_cache["s_hard_train"].shape,
        "s_soft_test": sensitive_label_cache["s_soft_test"].shape,
        "s_hard_test": sensitive_label_cache["s_hard_test"].shape,
    }
)
print(
    "Hard-positive rate (train/test):",
    float(sensitive_label_cache["s_hard_train"].mean()),
    float(sensitive_label_cache["s_hard_test"].mean()),
)

Sensitive labels ready from: cache -> /home/jupyter/claq/artifacts/sensitive_labels/cifar10
{'s_soft_train': (50000,), 's_hard_train': (50000,), 's_soft_test': (10000,), 's_hard_test': (10000,)}
Hard-positive rate (train/test): 0.5468999743461609 0.5491999983787537


In [6]:
def load_or_train_bundle(
    run_name,
    lambda_s,
    lambda_c,
    min_history=config.min_history,
    max_history=config.max_history,
    non_sensitive_only=config.non_sensitive_history_only,
    epochs=2,
    learning_rate=config.learning_rate,
    max_train_batches=60 if device.type == "cpu" else None,
    max_test_batches=30 if device.type == "cpu" else None,
    force_retrain=False,
):
    ckpt_path = paths.checkpoints_root / f"{run_name}_best.pt"
    history_path = paths.runs_root / f"{run_name}_history.json"
    if ckpt_path.exists() and not force_retrain:
        return load_run_bundle(ckpt_path, device=device, max_queries=config.max_queries, num_classes=config.num_classes)

    actor_checkpoint = str(paths.bootstrap_actor_checkpoint) if paths.bootstrap_actor_checkpoint.exists() else None
    classifier_checkpoint = str(paths.bootstrap_classifier_checkpoint) if paths.bootstrap_classifier_checkpoint.exists() else None

    actor, classifier, s_head = build_claq_models(
        max_queries=config.max_queries,
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
        actor_checkpoint=actor_checkpoint,
        classifier_checkpoint=classifier_checkpoint,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=min_history,
        max_history=max_history,
        non_sensitive_only=non_sensitive_only,
    )
    history, best = fit_claq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=train_loader,
        test_loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_s=lambda_s,
        lambda_c=lambda_c,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=epochs,
        max_train_batches=max_train_batches,
        max_test_batches=max_test_batches,
    )
    save_bundle_checkpoint(
        checkpoint_path=ckpt_path,
        metadata={
            "run_name": run_name,
            "lambda_s": lambda_s,
            "lambda_c": lambda_c,
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
            "history_config": {
                "min_history": history_config.min_history,
                "max_history": history_config.max_history,
                "non_sensitive_only": history_config.non_sensitive_only,
            },
            "actor_state_dict": best["actor_state_dict"],
            "classifier_state_dict": best["classifier_state_dict"],
            "s_head_state_dict": best["s_head_state_dict"],
        },
    )
    with open(history_path, "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    return load_run_bundle(ckpt_path, device=device, max_queries=config.max_queries, num_classes=config.num_classes)


baseline_bundle = load_or_train_bundle("baseline", lambda_s=0.0, lambda_c=0.0, epochs=5)
claq_bundle = load_or_train_bundle("lam_0.40", lambda_s=0.4, lambda_c=0.0, epochs=5)

print(baseline_bundle["ckpt_path"])
print(claq_bundle["ckpt_path"])

def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

/home/jupyter/claq/artifacts/models/baseline_best.pt
/home/jupyter/claq/artifacts/models/lam_0.40_best.pt


In [7]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    claq_bundle=claq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=raw_test_ds.classes,
    num_cases=4,
    pool_size=400 if device.type == "cpu" else 1500,
    num_trials=3,
    min_history=0,
    max_history=1,
    history_mode="non_sensitive",
    prefer_baseline_sensitive=True,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=figure_path("cifar10_intuition_replay_examples"),
    title_prefix="tiny-start replay",
)

print(f"Saved tiny-start replay figure: {intuition_fig}")

Sampling intuition replays:   0%|          | 0/1500 [00:00<?, ?it/s]

Saved tiny-start replay figure: /home/jupyter/claq/artifacts/figures/cifar10_intuition_replay_examples.svg


## Optional Comparison

For CIFAR-10, keep the aggregate comparison minimal.
Use this section only to compare `baseline` vs `lam=0.4` with fixed-history summary numbers.
We do not rely on sweep charts here because the aggregate story is weak and noisy.
If you later want a wider hyperparameter study, do it separately from the main CIFAR-10 notebook path.

In [8]:
comparison_name = "cifar10_baseline_vs_lam_0.4"
comparison_num_trials = 8
comparison_history_mode = "non_sensitive"
comparison_max_samples = 2000 if device.type == "cpu" else None

comparison_bundles = {
    "baseline": baseline_bundle,
    "lam=0.4": claq_bundle,
}

print("Using current main runs for CIFAR-10 comparison:")
print("- baseline ->", baseline_bundle["ckpt_path"])
print("- lam=0.4 ->", claq_bundle["ckpt_path"])

Using current main runs for CIFAR-10 comparison:
- baseline -> /home/jupyter/claq/artifacts/models/baseline_best.pt
- lam=0.4 -> /home/jupyter/claq/artifacts/models/lam_0.40_best.pt


In [9]:
comparison_fixed_history_rows = evaluate_bundles_on_fixed_histories(
    dataset=test_ds,
    answer_builder=answer_builder,
    bundles_by_name=comparison_bundles,
    sensitive_mask=sensitive_mask,
    min_history=config.min_history,
    max_history=config.max_history,
    history_mode=comparison_history_mode,
    num_trials=comparison_num_trials,
    max_samples=comparison_max_samples,
    eval_seed=config.random_seed,
    desc="Fixed-history comparison",
)

comparison_fixed_history_path = paths.runs_root / f"{comparison_name}_fixed_history_summary.json"
with open(comparison_fixed_history_path, "w", encoding="utf-8") as handle:
    json.dump(comparison_fixed_history_rows, handle, indent=2)

baseline_row = next(row for row in comparison_fixed_history_rows if row["run_name"] == "baseline")
baseline_acc = baseline_row["mean_acc"]
baseline_sens = baseline_row["mean_sensitive_query_rate"]

comparison_table = [
    {
        "run_name": row["run_name"],
        "lambda_s": row["lambda_s"],
        "mean_acc": round(row["mean_acc"], 4),
        "acc_delta_vs_baseline": round(row["mean_acc"] - baseline_acc, 4),
        "std_acc": round(row["std_acc"], 4),
        "mean_sensitive_query_rate": round(row["mean_sensitive_query_rate"], 4),
        "sens_delta_vs_baseline": round(row["mean_sensitive_query_rate"] - baseline_sens, 4),
        "std_sensitive_query_rate": round(row["std_sensitive_query_rate"], 4),
        "mean_confidence": round(row["mean_confidence"], 4),
    }
    for row in comparison_fixed_history_rows
]

comparison_table_path = paths.runs_root / f"{comparison_name}_summary_table.json"
with open(comparison_table_path, "w", encoding="utf-8") as handle:
    json.dump(comparison_table, handle, indent=2)

print(f"Saved fixed-history summary: {comparison_fixed_history_path}")
print(f"Saved comparison table: {comparison_table_path}")

comparison_table

Fixed-history comparison:   0%|          | 0/10000 [00:00<?, ?it/s]

Saved fixed-history summary: /home/jupyter/claq/artifacts/runs/cifar10_baseline_vs_lam_0.4_fixed_history_summary.json
Saved comparison table: /home/jupyter/claq/artifacts/runs/cifar10_baseline_vs_lam_0.4_summary_table.json


[{'run_name': 'baseline',
  'lambda_s': 0.0,
  'mean_acc': 0.3525,
  'acc_delta_vs_baseline': 0.0,
  'std_acc': 0.0044,
  'mean_sensitive_query_rate': 0.031,
  'sens_delta_vs_baseline': 0.0,
  'std_sensitive_query_rate': 0.0021,
  'mean_confidence': 0.3877},
 {'run_name': 'lam=0.4',
  'lambda_s': 0.4,
  'mean_acc': 0.3508,
  'acc_delta_vs_baseline': -0.0017,
  'std_acc': 0.0049,
  'mean_sensitive_query_rate': 0.0248,
  'sens_delta_vs_baseline': -0.0062,
  'std_sensitive_query_rate': 0.0015,
  'mean_confidence': 0.3858}]